In [2]:
import pandas as pd
from main import SatCLIPLightningModule
import location_encoder as LE
from temporal_encoding import Fourier, Direct


/home/leca5365/miniconda3/envs/satclip313/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Load the GHCN dataset
ghcn_df = pd.read_csv("../notebooks/ghcn_2020_2024_with_stations.csv")

In [3]:
# Load SatCLIP model
ckpt_path = '/home/leca5365/Documents/satclip/satclip/satclip_temporal_logs/satclip-s2-temporal-55k/satclip-fixed-fourier/checkpoints/best.ckpt'
lightning_model = SatCLIPLightningModule.load_from_checkpoint(ckpt_path)

lightning_model.eval()
spatiotemporal_enc = lightning_model.model.location
visual_enc = lightning_model.model.visual

using pretrained moco vit16


In [ ]:
# create MLP for temperature prediction
import torch
import torch.nn as nn

class TemperaturePredictor(nn.Module):
    def __init__(self, location_encoder, hidden_dim=256):
        super(TemperaturePredictor, self).__init__()
        self.location_encoder = location_encoder
        self.fc1 = nn.Linear(location_encoder.output_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 1)  # Predict temperature

    def forward(self, lat, lon, time, image):
        loc_emb = self.location_encoder([lat, lon, time])
        x = torch.relu(self.fc1(loc_emb))
        temp_pred = self.fc2(x)
        return temp_pred
    
tp = TemperaturePredictor(spatiotemporal_enc)

for param in tp.location_encoder.parameters():
    param.requires_grad = False

opt = torch.optim.Adam(tp.parameters(), lr=1e-3)
criterion = nn.MSELoss()

from torch.utils.data import DataLoader, TensorDataset

# Prepare dataset
def prepare_dataset(df):
    latitudes = torch.tensor(df['latitude'].values, dtype=torch.float32)
    longitudes = torch.tensor(df['longitude'].values, dtype=torch.float32)
    times = torch.tensor(pd.to_datetime(df['date']).astype(int) / 10**9, dtype=torch.float32)  # Convert to timestamp
    temperatures = torch.tensor(df['temperature'].values, dtype=torch.float32).unsqueeze(1)
    
    dataset = TensorDataset(latitudes, longitudes, times, temperatures)
    return dataset
